In [ ]:
# dependencies
# uv add anthropic python-dotenv
# python -m pip install ipykernel -U --force-reinstall

UsageError: Line magic function `%poetry` not found.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [8]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role" : "user",
            "content" : "What is quantum computing? Answer in one sentence."
        }
    ]
)

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CYcDiD62aj7TJU1M6P4xq'}

# COMET API

In [ ]:
import http.client
import json

conn = http.client.HTTPSConnection("api.cometapi.com")
payload = json.dumps({
   "max_tokens": 1024,
   "system": "You are a helpful assistant.",
   "messages": [
      {
         "content": "Hello, world",
         "role": "user"
      }
   ],
   "model": "claude-sonnet-4-5-20250929"
})
headers = {
   'Authorization': 'Bearer ',
   'anthropic-beta': 'compact-2026-01-12',
   'Content-Type': 'application/json'
}
conn.request("POST", "/v1/messages", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

{"error":{"type":"comet_api_error","message":"user quota is not enough, remaining quota: ＄0.000000 (request id: 20260301185307999252847tCdkI5zX)"},"type":"error"}


In [8]:
import http.client

conn = http.client.HTTPSConnection("api.cometapi.com")
payload = ''
headers = {}
conn.request("GET", "/api/models", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

{"data":[{"created":1772155021,"id":"gemini-3.1-flash-image-preview","code":"gemini-3-1-flash-image-preview","provider":"Google","provider_code":"google","provider_sort":98,"model_sort":1765856092,"description":"Core Capabilities Overview: Resolution: Up to 4K (4096×4096), on par with Pro. Reference Image Consistency: Up to 14 reference images (10 objects + 4 characters), maintaining style/character consistency. Extreme Aspect Ratios: New 1:4, 4:1, 1:8, 8:1 ratios added, suitable for long images, posters, and banners. Text Rendering: Advanced text generation, suitable for infographics and marketing poster layouts. Search Enhancement: Integrated Google Search + Image Search. Grounding: Built-in thinking process; complex prompts are reasoned before generation.","tags":[],"name":"Nano Banana 2","model_type":"image","context_length":null,"max_completion_tokens":null,"features":["text-to-image","image-editing"],"endpoints":"{\n  \"gemini\": {\n    \"path\": \"/v1beta/models/{model}:generate

In [5]:
import http.client
import json

conn = http.client.HTTPSConnection("api.cometapi.com")
payload = json.dumps({
   "model": "gpt-4.1",
   "messages": [
      {
         "role": "system",
         "content": "You are a helpful assistant."
      },
      {
         "role": "user",
         "content": "Hello!"
      }
   ]
})
headers = {
   'Authorization': f'Bearer {os.getenv("COMET_API_KEY")}',
   'Content-Type': 'application/json'
}
conn.request("POST", "/v1/chat/completions", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

{"error":{"message":"user quota is not enough, remaining quota: ＄0.000000 (request id: 20260301180424786976451s28k4Wud)","type":"comet_api_error","param":"","code":"insufficient_user_quota"}}


# AI generated puter

In [6]:
import requests

def call_free_ai_model(prompt: str, model: str = "gpt-4o-mini") -> str:
    """
    Calls a free AI API endpoint and returns the generated text.
    
    Args:
        prompt (str): The text prompt to send to the AI model.
        model (str): The model name (default: gpt-4o-mini).
    
    Returns:
        str: The AI-generated response.
    """
    try:
        # Public free AI API endpoint (Puter AI)
        url = "https://api.puter.com/v2/ai/chat/completions"
        
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        }
        
        headers = {"Content-Type": "application/json"}
        
        response = requests.post(url, json=payload, headers=headers, timeout=15)
        response.raise_for_status()  # Raise error for HTTP issues
        
        data = response.json()
        
        # Extract AI's reply
        return data.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
    
    except requests.exceptions.RequestException as e:
        return f"Network/API error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

# Example usage
if __name__ == "__main__":
    user_prompt = "Write a short poem about the ocean."
    ai_reply = call_free_ai_model(user_prompt)
    print("AI Response:\n", ai_reply)


AI Response:
 Network/API error: 403 Client Error: Forbidden for url: https://api.puter.com/v2/ai/chat/completions


# Openrouter

In [ ]:
import requests
import json
from dotenv import load_dotenv
import os

load_dotenv()

class LLM:
    messages = [{
        "role" : "user",
        "content" : "You are a medical chatbot and you are not supposed to reply to questions from any other domain and should reply Sorry I do not know about that",
    }]
    # messages = []

    @classmethod
    def add_user_message(cls, text):
        user_message = {
            "role" : "user",
            "content" : text
        }
        cls.messages.append(user_message)

    @classmethod
    def add_assistant_message(cls, text):
        assistant_message = {
            "role" : "assistant",
            "content" : text,
        }
        cls.messages.append(assistant_message)

    @classmethod
    def chat(cls, message):
        cls.add_user_message(message)
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            },
            data=json.dumps({
                "model": "nvidia/nemotron-3-nano-30b-a3b:free",
                "messages": cls.messages,
                "system" : "You are a medical chatbot and you are not supposed to reply to questions from any other domain and should reply Sorry I do not know about that"
            })
        )
        llm_response = response.json().get('choices')[0].get('message').get('content')
        cls.add_assistant_message(llm_response)
        return llm_response


In [42]:
user_input = input("Please enter a prompt. Enter 'q' to quit.")
while user_input!='q':
    print(LLM.chat(user_input))
    user_input = input("Please enter a prompt. Enter 'q' to quit.")

**Quick 3‑Step Rice Method (1 cup uncooked rice, serves 2‑3)**  

1. **Rinse & Measure**  
   - Place rice in a fine‑mesh sieve. Rinse under cold water, swishing until the water runs clear (about 30 seconds). Drain well.  
   - Combine the rinsed rice with 1 ½ cups water (or broth) in a pot. Add a pinch of salt if you like.

2. **Cook**  
   - Bring to a rolling boil over medium‑high heat.  
   - As soon as it boils, stir once, then immediately reduce heat to **low**, cover tightly, and **simmer 12–15 minutes** (no peeking!).

3. **Rest & Fluff**  
   - Turn off the heat and let the pot sit, still covered, **5 minutes**.  
   - Remove the lid, fluff the rice with a fork, and serve.

**Tips in a nutshell**  
- **Ratio**: 1 cup rice : 1½ cups water (standard white/long‑grain).  
- **No stirring** while it’s simmering—keeps the rice from getting sticky.  
- If you’re in a hurry, use a microwave or rice cooker with the same 1:1½ ratio; just follow the appliance’s timing.  

*That’s it—done

In [40]:
LLM.messages

[{'role': 'user',
  'content': 'You are a medical chatbot and you are not supposed to reply to questions from any other domain and should reply Sorry I do not know about that'},
 {'role': 'user', 'content': 'how to cook rice in short'},
 {'role': 'assistant', 'content': 'Sorry I do not know about that'},
 {'role': 'user', 'content': 'how is perianal abscessa'},
 {'role': 'assistant',
  'content': 'A perianal abscess is a collection of pus that forms just outside the anus, usually as a result of an infected anal gland. Common signs include:\n\n- Persistent pain or throbbing near the anus, especially when sitting or during bowel movements  \n- Swelling, redness, and a visible lump or “boil” in the area  \n- Possible drainage of pus or foul‑smelling fluid  \n- Fever or feeling generally unwell in more severe cases  \n\nTreatment typically involves draining the abscess (often by a healthcare professional) and, if needed, a course of antibiotics to clear any infection. Prompt medical evalua

In [ ]:


response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": "Bearer ",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3-nano-30b-a3b:free", # Optional
    "messages": [
      {
        "role": "user",
        "content": "What is the meaning of life in one sentence?"
      }
    ]
  })
)
print(response.json().get('choices')[0].get('message').get('content'))


The meaning of life is to purposefully experience, connect with others, and grow through our joys and challenges, turning each moment into insight and compassion.
